In [0]:
# =============================================================================
# FILE:    silver_utils.py
# PURPOSE: Shared utility functions used by ALL Silver layer notebooks.
#          Import this file after silver_config.py in every Silver notebook.
#
# WHY A SHARED UTILS FILE?
#   The same operations appear in every Silver notebook:
#     - Delta MERGE (upsert) to prevent duplicates
#     - Data quality validation (drop bad rows, log drop rate)
#     - Writing a JSON run log to ADLS
#   Without utils.py, each notebook would copy-paste these ~80 lines.
#   Copy-paste = bugs diverging silently across notebooks. One utils file
#   means one fix propagates everywhere.
#
# FUNCTIONS:
#   delta_merge()         → upsert a DataFrame into a Silver Delta table
#   validate_drop_rate()  → check if too many rows were dropped by quality filters
#   write_run_log()       → persist a JSON summary of the run to ADLS logging/
#   get_silver_timestamp()→ return current UTC timestamp as a Spark Column
#   read_bronze_latest()  → read the most recent pipeline_run_id from a Bronze table
#   log_schema()          → log DataFrame schema to Python logger for debugging


In [0]:
import json
import logging

from typing import List, Dict, Any, Optional

from pyspark.sql            import DataFrame, SparkSession
from pyspark.sql            import functions as F
from pyspark.sql.types      import TimestampType
from delta.tables           import DeltaTable

In [0]:
# =============================================================================
# LOGGING SETUP
# Each notebook that imports this file gets its own named logger.
# The notebook passes its own name so log messages identify the source notebook.
# =============================================================================

def get_logger(notebook_name: str) -> logging.Logger:
    """
    Return a configured Python logger for a Silver notebook.
    Call once at the top of each notebook:
        logger = get_logger("silver_market_snapshot")
    """
    logger = logging.getLogger(f"silver.{notebook_name}")
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter(
            "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
            datefmt="%Y-%m-%dT%H:%M:%SZ"
        ))
        logger.addHandler(handler)
        logger.setLevel(logging.INFO)
    return logger



In [0]:
# =============================================================================
# FUNCTION: read_bronze_latest()
#
# WHY THIS FUNCTION EXISTS:
#   Each Bronze Delta table accumulates one batch of rows per daily pipeline run.
#   When Silver runs, it should process ONLY the latest pipeline run's rows,
#   not re-process data from previous days (which would already be in Silver).
#
#   We filter Bronze by the MAX(pipeline_run_id) value — but pipeline_run_id
#   is a UUID string, not a sortable timestamp. Instead we use bronze_loaded_at
#   (the timestamp we added in the Bronze Autoloader) to find the most recent batch.
#
#   This gives us the "today's batch" from Bronze without needing a checkpoint.
# =============================================================================

def read_bronze_latest(
    spark        : SparkSession,
    bronze_path  : str,
    logger       : logging.Logger,
) -> DataFrame:
    """
    Read the most recent pipeline run's rows from a Bronze Delta table.
    Identifies the latest batch by MAX(bronze_loaded_at) and filters to that batch.

    WHY NOT READ ALL BRONZE ROWS?
      Bronze accumulates rows across all runs. Re-reading all of them every day
      would reprocess yesterday's data, and the MERGE would mark them as "already
      exists → skip". While safe, it wastes Spark resources and slows Silver down.
      Reading only the latest batch is efficient AND produces the same result.

    Args:
        spark       : Active SparkSession
        bronze_path : Full abfss:// path to the Bronze Delta table
        logger      : Caller's logger for consistent log formatting

    Returns:
        DataFrame containing only the most recent pipeline run's rows
    """
    logger.info(f"  Reading Bronze: {bronze_path}")

    # Read the full Bronze table (Delta handles file skipping via _delta_log)
    bronze_df = spark.read.format("delta").load(bronze_path)

    # Find the maximum bronze_loaded_at timestamp — this represents today's batch
    max_ts_row = bronze_df.agg(F.max("bronze_loaded_at").alias("max_ts")).collect()[0]
    max_ts     = max_ts_row["max_ts"]

    logger.info(f"  Latest bronze_loaded_at: {max_ts}")

    # Filter to only this batch's rows
    latest_df = bronze_df.filter(F.col("bronze_loaded_at") == max_ts)

    count = latest_df.count()
    logger.info(f"  Latest batch row count: {count:,}")

    return latest_df

In [0]:
# =============================================================================
# FUNCTION: delta_merge()
#
# WHY MERGE INSTEAD OF OVERWRITE?
#   .mode("overwrite") would delete ALL existing Silver rows every run,
#   losing historical data. .mode("append") would add duplicate rows if the
#   pipeline re-runs the same day (e.g., due to a bug fix or retry).
#
#   Delta MERGE is the correct pattern:
#     - If a row with the same key already exists in Silver → SKIP (not matched)
#     - If the row is genuinely new → INSERT it
#   This makes Silver idempotent: running it 10 times produces the same result
#   as running it once.
#
# MERGE CONDITION BUILDING:
#   merge_keys = ["coin_id", "snapshot_date"]
#   → condition = "existing.coin_id = new.coin_id AND existing.snapshot_date = new.snapshot_date"
# =============================================================================

def delta_merge(
    spark       : SparkSession,
    new_df      : DataFrame,
    table_path  : str,
    merge_keys  : List[str],
    logger      : logging.Logger,
) -> Dict[str, Any]:
    """
    Upsert new_df into an existing Silver Delta table using Delta MERGE.
    If the table does not yet exist, creates it on the first run.

    Args:
        spark      : Active SparkSession
        new_df     : Transformed Silver DataFrame to merge in
        table_path : Full abfss:// path to the Silver Delta table
        merge_keys : List of column names that form the primary key
        logger     : Caller's logger

    Returns:
        Dict with "rows_before", "rows_after", "rows_inserted" counts
    """
    # Build the MERGE condition string
    # e.g., "existing.coin_id = new.coin_id AND existing.snapshot_date = new.snapshot_date"
    merge_condition = " AND ".join(
        [f"existing.{col} = new.{col}" for col in merge_keys]
    )
    logger.info(f"  MERGE into: {table_path}")
    logger.info(f"  Merge condition: {merge_condition}")

    # Check if the Delta table exists yet
    table_exists = DeltaTable.isDeltaTable(spark, table_path)

    if not table_exists:
        # First run: no table yet → just write directly
        # This creates the Delta table with the correct schema.
        # All subsequent runs will use MERGE.
        logger.info("  Table does not exist yet — creating on first write")
        (
            new_df
            .write
            .format("delta")
            .mode("overwrite")    # Safe here: first write, nothing to lose
            .option("mergeSchema", "true")
            .save(table_path)
        )
        rows_after = new_df.count()
        logger.info(f"  ✓ Table created | rows: {rows_after:,}")
        return {"rows_before": 0, "rows_after": rows_after, "rows_inserted": rows_after}

    # Table exists → use MERGE (upsert)
    delta_table = DeltaTable.forPath(spark, table_path)

    # Count before for the summary
    rows_before = delta_table.toDF().count()

    (
        delta_table.alias("existing")
        .merge(
            new_df.alias("new"),
            merge_condition
        )
        # Only insert rows that don't already exist — we never UPDATE Silver rows
        # because Bronze is the source of truth; if you need corrections, reprocess
        # from Bronze with a new pipeline_run_id.
        .whenNotMatchedInsertAll()
        .execute()
    )

    rows_after   = delta_table.toDF().count()
    rows_inserted = rows_after - rows_before

    logger.info(f"  ✓ MERGE complete | before: {rows_before:,} | after: {rows_after:,} | inserted: {rows_inserted:,}")
    return {
        "rows_before"  : rows_before,
        "rows_after"   : rows_after,
        "rows_inserted": rows_inserted,
    }



In [0]:
# =============================================================================
# FUNCTION: validate_drop_rate()
#
# WHY THIS CHECK?
#   Data quality filters (e.g., drop rows where price is null or <= 0) are
#   expected to remove a SMALL fraction of rows. If 35% of a batch is being
#   dropped, something is wrong upstream — maybe the API changed its format,
#   or a new coin type has different nullability. We want to catch this loudly
#   rather than silently let bad data propagate to Gold.
# =============================================================================

def validate_drop_rate(
    rows_before  : int,
    rows_after   : int,
    max_fraction : float,
    table_name   : str,
    logger       : logging.Logger,
) -> None:
    """
    Check that data quality filters did not drop more than max_fraction of rows.
    Raises DataQualityError if exceeded.

    Args:
        rows_before  : Row count before quality filters
        rows_after   : Row count after quality filters
        max_fraction : Maximum acceptable drop fraction (e.g., 0.30 = 30%)
        table_name   : Name for error messages
        logger       : Caller's logger
    """
    if rows_before == 0:
        logger.warning(f"  [{table_name}] Input batch has 0 rows — nothing to validate")
        return

    dropped  = rows_before - rows_after
    fraction = dropped / rows_before

    logger.info(f"  [{table_name}] Quality filter: {dropped:,} rows dropped ({fraction:.1%} of {rows_before:,})")

    if fraction > max_fraction:
        msg = (
            f"Data quality check FAILED for {table_name}: "
            f"{fraction:.1%} of rows dropped (max allowed: {max_fraction:.1%}). "
            f"rows_before={rows_before:,}, rows_after={rows_after:,}. "
            f"Investigate Bronze source data before proceeding."
        )
        logger.error(f"  {msg}")
        raise ValueError(msg)

In [0]:
# =============================================================================
# FUNCTION: write_run_log()
#
# WHY PERSIST RUN LOGS?
#   Databricks notebook logs live only as long as the cluster session.
#   Once the cluster is terminated, those logs are gone. Writing a structured
#   JSON summary to ADLS means:
#     - Azure Monitor / Log Analytics can ingest and alert on them
#     - You can query run history across all days using Spark
#     - If a Gold table looks wrong, you can trace it back to the Silver run
#       that produced the data using pipeline_run_id
# =============================================================================

def write_run_log(
    summary    : Dict[str, Any],
    log_path   : str,
    logger     : logging.Logger,
) -> None:
    """
    Write a JSON run summary to the ADLS logging container.
    Uses dbutils.fs.put (available in all Databricks notebooks).

    Args:
        summary  : Dict containing run metadata and results
        log_path : Full abfss:// path for the log file
        logger   : Caller's logger
    """
    try:
        log_content = json.dumps(summary, indent=2, default=str)
        dbutils.fs.put(log_path, log_content, overwrite=True)
        logger.info(f"  ✓ Run log written → {log_path}")
    except Exception as e:
        # Log writing failure should NEVER fail the pipeline.
        # The data is already in Silver — losing a log entry is acceptable.
        logger.warning(f"  ⚠ Failed to write run log (non-fatal): {e}")


# =============================================================================
# FUNCTION: get_silver_timestamp()
#
# Returns the current UTC timestamp as a PySpark Column for use in:
#   df.withColumn("silver_processed_timestamp", get_silver_timestamp())
#
# WHY A HELPER?
#   current_timestamp() returns the local cluster time, which may not be UTC.
#   We explicitly use UTC by passing the notebook's RUN_TS value (captured
#   at notebook start) as a fixed literal cast to TimestampType.
#   This ensures all rows in a Silver run share the same processed timestamp.
# =============================================================================

def get_silver_timestamp(run_ts) -> "Column":
    """
    Return a Spark Column with the fixed UTC run timestamp as TimestampType.
    Use as: df.withColumn("silver_processed_timestamp", get_silver_timestamp(RUN_TS))

    Args:
        run_ts: datetime object (from SilverConfig.RUN_TS)
    Returns:
        PySpark Column of TimestampType
    """
    return F.lit(run_ts.isoformat()).cast(TimestampType())



In [0]:
# =============================================================================
# FUNCTION: log_schema()
# Quick debug helper to print a DataFrame's schema through the logger.
# =============================================================================

def log_schema(df: DataFrame, label: str, logger: logging.Logger) -> None:
    """Print the DataFrame schema to the logger at DEBUG level."""
    schema_str = df.schema.simpleString()
    logger.debug(f"  Schema [{label}]: {schema_str}")


# =============================================================================
# FUNCTION: assert_required_columns()
# Defensive check before transformation: ensure Bronze contains the columns
# we expect. If CoinGecko changes their API and a field disappears, this
# gives a clear error message instead of a cryptic Spark AnalysisException.
# =============================================================================

def assert_required_columns(
    df              : DataFrame,
    required_cols   : List[str],
    table_name      : str,
    logger          : logging.Logger,
) -> None:
    """
    Assert that all required columns exist in the DataFrame.
    Raises ValueError with a clear message if any are missing.

    Args:
        df            : DataFrame to check
        required_cols : List of column names that must be present
        table_name    : Name for error messages
        logger        : Caller's logger
    """
    existing = set(df.columns)
    missing  = set(required_cols) - existing

    if missing:
        msg = (
            f"Missing required columns in {table_name}: {sorted(missing)}. "
            f"Available columns: {sorted(existing)}. "
            f"Check if the CoinGecko API response format has changed."
        )
        logger.error(f"  {msg}")
        raise ValueError(msg)

    logger.info(f"  ✓ All {len(required_cols)} required columns present in {table_name}")